[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Red1-Rahman/NiriZan/blob/main/experiments/01_instrumentation_trace_storage.ipynb)
[![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://www.kaggle.com/kernels/welcome?src=https://github.com/Red1-Rahman/NiriZan/blob/main/experiments/01_instrumentation_trace_storage.ipynb)

# Experiment 01: Instrumentation & Trace Storage
**Phase 1 Exploration**: Validating asynchronous span emission, immutable trace creation, schema constraints, and storage repository interfaces under simulated application loads.

## 1. Environment Setup
Install `nirizan` directly from the main branch or setup local dev dependencies.

In [3]:
import sys

# Install NiriZan and dependencies silently when running in Colab/Kaggle
!pip install -q pydantic>=2.7 "git+https://github.com/Red1-Rahman/NiriZan.git@main#egg=nirizan"

from datetime import datetime, timezone
import time
from uuid import uuid4
import asyncio

from nirizan.instrumentation.spans import Span, SpanKind, Trace
from pydantic import ValidationError

print("✅ NiriZan successfully loaded!")

✅ NiriZan successfully loaded!


## 2. Validating `Span` Model Contracts
Spans are the atomic unit of instrumentation[cite: 6]. We test:
1. Span creation across all `SpanKind` enums (`PLANNING`, `RETRIEVAL`, `TOOL_USE`, `GENERATION`)[cite: 6].
2. Strict frozen immutability (`frozen=True`)[cite: 6].
3. Attribute primitive typing constraints[cite: 6].

In [4]:
trace_id = uuid4()
now = datetime.now(timezone.utc)

# 1. Create a retrieval span
retrieval_span = Span(
    span_id=uuid4(),
    trace_id=trace_id,
    kind=SpanKind.RETRIEVAL,
    name="qdrant_vector_search",
    started_at=now,
    ended_at=now,
    attributes={"top_k": 5, "vector_dim": 1536, "hybrid_search": True},
    input_payload="What is continuous evaluation?",
    output_payload="Doc 1: Continuous evaluation infrastructure...",
)

print(f"Created Span ID: {retrieval_span.span_id}")
print(f"Attributes: {retrieval_span.attributes}")

# 2. Test Immutability
try:
    retrieval_span.name = "modified_name"  # type: ignore
except ValidationError as e:
    print("\n✅ Immutability verified: Cannot modify frozen Span instance!")

Created Span ID: 8a011f79-129a-4ed4-b747-030b04e6c03e
Attributes: {'top_k': 5, 'vector_dim': 1536, 'hybrid_search': True}

✅ Immutability verified: Cannot modify frozen Span instance!


## 3. Assembling a `Trace`
A `Trace` is an ordered collection of spans belonging to a single application invocation[cite: 6].
We test:
1. `trace_id` validation across child spans[cite: 6].
2. Filtering spans by `SpanKind` via `spans_of_kind()`[cite: 6].

In [5]:
generation_span = Span(
    span_id=uuid4(),
    trace_id=trace_id,
    kind=SpanKind.GENERATION,
    name="llm_generate_answer",
    started_at=now,
    ended_at=now,
    attributes={"model": "gpt-4o", "temperature": 0.2},
    input_payload="Context: ... Prompt: What is continuous evaluation?",
    output_payload="Continuous evaluation is an engineering layer...",
)

# Create Trace
trace = Trace(
    trace_id=trace_id,
    application_name="production_rag_service",
    spans=[retrieval_span, generation_span],
    created_at=now,
)

print(f"Trace ID: {trace.trace_id}")
print(f"Total Spans: {len(trace.spans)}")
print(f"Retrieval Spans: {len(trace.spans_of_kind(SpanKind.RETRIEVAL))}")
print(f"Generation Spans: {len(trace.spans_of_kind(SpanKind.GENERATION))}")

# Test trace_id mismatch assertion
mismatched_span = Span(
    span_id=uuid4(),
    trace_id=uuid4(),  # Different trace_id!
    kind=SpanKind.PLANNING,
    name="query_planner",
    started_at=now,
    ended_at=now,
)

try:
    Trace(
        trace_id=trace_id,
        application_name="invalid_trace_app",
        spans=[mismatched_span],
        created_at=now,
    )
except ValidationError:
    print("\n✅ Trace ID validation verified: Rejected mismatched span!")

Trace ID: 7b3d78e1-77fe-4d39-a2a1-f1b16f379edc
Total Spans: 2
Retrieval Spans: 1
Generation Spans: 1

✅ Trace ID validation verified: Rejected mismatched span!


## 4. Measuring Tracing Overhead (Latency Benchmark)
Instrumentation must **never** block the application's request/response path[cite: 6].
We simulate async background trace exportation to verify minimal latency impact.

In [6]:
class MockAsyncExporter:
    """Simulates an asynchronous background collector export."""

    async def export(self, trace: Trace) -> None:
        # Simulate background network/storage latency without blocking caller
        await asyncio.sleep(0.05)


async def simulate_application_request(exporter: MockAsyncExporter):
    start_time = time.perf_counter()

    # Application execution simulation
    t_id = uuid4()
    n = datetime.now(timezone.utc)
    s = Span(
        span_id=uuid4(),
        trace_id=t_id,
        kind=SpanKind.GENERATION,
        name="rag_response",
        started_at=n,
        ended_at=n,
    )
    tr = Trace(
        trace_id=t_id, application_name="benchmark_app", spans=[s], created_at=n
    )

    # Fire-and-forget background task for trace emission
    asyncio.create_task(exporter.export(tr))

    elapsed_ms = (time.perf_counter() - start_time) * 1000
    return elapsed_ms


async def run_benchmark():
    exporter = MockAsyncExporter()
    latencies = []

    for _ in range(1000):
        latency = await simulate_application_request(exporter)
        latencies.append(latency)

    avg_latency = sum(latencies) / len(latencies)
    print(f"⚡ Tracing Latency Overhead across 1,000 runs:")
    print(f"   Average Overhead: {avg_latency:.4f} ms per request")
    print(
        f"   Max Single-Request Overhead: {max(latencies):.4f} ms (Non-blocking verified)"
    )


await run_benchmark()

⚡ Tracing Latency Overhead across 1,000 runs:
   Average Overhead: 0.0154 ms per request
   Max Single-Request Overhead: 1.6274 ms (Non-blocking verified)


Redwan Rahman